# PixelTable

In [1]:
# Import packages
import os
import tempfile

import polars as pl
import hicona as hi

# Data paths 
COOLER_PATH = "../data/IMR90_100kb_chr17_balanced.cool"
HMM_STATES = "../data/hmm_states_chr17.bed"
RANDINTS = "../data/rand_ints_chr17:0-10000000.bed"

## Creating a PixelTable

The `PixelTable` is the main class around which HiCONA revolves. If you are not planning on extending HiCONA's functionalities or trying niche corner-cases, once you have created a `PixelTable` object you will likely work exclusively on it. 

A `PixelTable` contains pixels in the `bin1_id`, `bin2_id`, `count` format (with optional annotation columns). Each `PixelTable` is associated with a `BinTable`. The `BinTable` contains the genomic binning used, meanining that it has the columns `bin_id`, `chrom`, `start`, `end` plus any number of bin level annotation columns. The way `PixelTable` and `BinTable` instances handle memory is explained in the `Implementation` section of the documentation.

<div class=note>
    Though `BinTable` is a standalone class, it will almost always be used indirectly via a `PixelTable` instance. For simplicity sake,
    throughout this documentation, `PixelTable` is always accessed as a `PixelTable` attribute. For more detail on `BinTable`, please refer
    to the API documentation directly.
</div>

<div class=warn>
Internally, HiCONA uses <a href="https://pola.rs/">polars</a> to handle datasets for better performance. Since polars does not support index columns, the `bin_id` column in the `BinTable` is an explicit column, rather than the index itself. When fetching dataframes in pandas, HiCONA provides no guarantee that the index matches the `bin_id` column.
</div>

The main way you will create a `PixelTable` is from a `HiconaCooler` object by specifying a genomic region (the retrieved pixels are those for which *both* bins fall within the specified interval).

In [2]:
handle = hi.HiconaCooler(COOLER_PATH)
table = handle.get_pixel_table("chr17")

This step might be a bit slow for very huge files (10s of seconds), since a full pass of the cooler is required and the data needs to be written to the temporary storage. The advantage is that those pixels now become and stay adjacent into memory, allowing to apply complex operations, adding annotations, and performing multiple iterations quickly and without affecting the original file. 

When creating a `PixelTable` this way, a `BinTable` is also created indirectly and saved to the `bins` attribute of the `PixelTable` instance. The `BinTable` will always contain the binning of the entire genome, even if a subsetting region is specified. This is because table comparison and plotting can become very difficult with partial binning.

In [3]:
table.bins

Both `BinTable` and `PixelTable` instances can also be created directly from iterables of bins and pixels (similar to `cooler.create_cooler`).

## Reading Data

The `PixelTable` class provides three methods to access its data: `get_chunks`, `get_dataframe`, `get_matrix`.

`get_chunks`, as the name suggests, returns an generator of chunks of the `PixelTable`. The full table can be returned or a subset of it by providing a genomic region. Furthermore, different parameters can be returned to customize these chunks:

- `annotate`: whether to return `bin1_id`, `bin2_id`, `count` or the expended format (see `cooler.annotate`).
- `balance`: apply ICE normalization. Requires a `weight` column to be present in the `BinTable`.
- `chunk_size`: dimension of the chunks to fetch.
- `dtype`: the type of dataframe returned (`pandas` or `polars`)
- `strategies`: any user defined function which takes a generator of chunks as input and returns a new one as output.

In [4]:
# Default chunks
for chunk in table.get_chunks():
    print(chunk.head())
    break

shape: (5, 3)
┌─────────┬─────────┬───────┐
│ bin1_id ┆ bin2_id ┆ count │
│ ---     ┆ ---     ┆ ---   │
│ i64     ┆ i64     ┆ i32   │
╞═════════╪═════════╪═══════╡
│ 26475   ┆ 26475   ┆ 17    │
│ 26475   ┆ 26481   ┆ 1     │
│ 26475   ┆ 26518   ┆ 1     │
│ 26475   ┆ 26526   ┆ 1     │
│ 26475   ┆ 26534   ┆ 1     │
└─────────┴─────────┴───────┘


In [5]:
# Region subset and annotated chunks"
for chunk in table.get_chunks("chr17:0-1000000", annotate=True):
    print(chunk.head())
    break

shape: (5, 11)
┌─────────┬─────────┬───────┬────────┬───┬────────┬────────┬────────┬──────────┐
│ bin1_id ┆ bin2_id ┆ count ┆ chrom1 ┆ … ┆ chrom2 ┆ start2 ┆ end2   ┆ weight2  │
│ ---     ┆ ---     ┆ ---   ┆ ---    ┆   ┆ ---    ┆ ---    ┆ ---    ┆ ---      │
│ i64     ┆ i64     ┆ i32   ┆ str    ┆   ┆ str    ┆ i32    ┆ i32    ┆ f64      │
╞═════════╪═════════╪═══════╪════════╪═══╪════════╪════════╪════════╪══════════╡
│ 26475   ┆ 26475   ┆ 17    ┆ chr17  ┆ … ┆ chr17  ┆ 0      ┆ 100000 ┆ null     │
│ 26475   ┆ 26481   ┆ 1     ┆ chr17  ┆ … ┆ chr17  ┆ 600000 ┆ 700000 ┆ 0.004789 │
│ 26476   ┆ 26476   ┆ 7035  ┆ chr17  ┆ … ┆ chr17  ┆ 100000 ┆ 200000 ┆ 0.007338 │
│ 26476   ┆ 26477   ┆ 4003  ┆ chr17  ┆ … ┆ chr17  ┆ 200000 ┆ 300000 ┆ 0.006855 │
│ 26476   ┆ 26478   ┆ 3392  ┆ chr17  ┆ … ┆ chr17  ┆ 300000 ┆ 400000 ┆ 0.005692 │
└─────────┴─────────┴───────┴────────┴───┴────────┴────────┴────────┴──────────┘


In [6]:
# Pandas chunks of size 500"
for chunk in table.get_chunks(dtype="pandas", chunk_size=500):
    print(chunk.head())
    break

   bin1_id  bin2_id  count
0    26475    26475     17
1    26475    26481      1
2    26475    26518      1
3    26475    26526      1
4    26475    26534      1


In [7]:
# Custom strategy chunks
count_scaler = lambda x: (y.with_columns(pl.col("count") * 10) for y in x)  # Multiply counts by 10
for chunk in table.get_chunks(strategies=count_scaler):
    print(chunk.head())
    break

shape: (5, 3)
┌─────────┬─────────┬───────┐
│ bin1_id ┆ bin2_id ┆ count │
│ ---     ┆ ---     ┆ ---   │
│ i64     ┆ i64     ┆ i32   │
╞═════════╪═════════╪═══════╡
│ 26475   ┆ 26475   ┆ 170   │
│ 26475   ┆ 26481   ┆ 10    │
│ 26475   ┆ 26518   ┆ 10    │
│ 26475   ┆ 26526   ┆ 10    │
│ 26475   ┆ 26534   ┆ 10    │
└─────────┴─────────┴───────┘


`get_dataframe` is simply `get_chunks` with concatenation on top. `region`, `annotate` and `dtype` parameters are natively supported by the method. All other parameters to pass to the internal call to `get_chunks` should be provided as a dictionary in the `selection_kwargs` parameter. `get_dataframe` also works on `BinTable` objects.

In [8]:
table.get_dataframe("chr17:1000000-2000000").head()

bin1_id,bin2_id,count
i64,i64,i32
26485,26485,16700
26485,26486,5021
26485,26487,1164
26485,26488,607
26485,26489,434


`get_matrix` is instead used to return a 2D-numpy array. Again, this supports the `region` parameter as well as `selection_kwargs`, but it also provides method-specific parameters, namely:

- `value_col`: the values to put in the cells of the matrix. By default `counts` but could be clustering labels and others.
- `mode`: whether to return a full, upper triangular or lower triangular matrix.
- `mask_diagonal`: whether to fill the main diagonal with nans.
        

In [9]:
matrix = table.get_matrix("chr17:0-10000000", mode="upper")
print(matrix)

[[1.7000e+01        nan        nan ...        nan        nan        nan]
 [       nan 7.0350e+03 4.0030e+03 ... 1.4000e+01 1.3000e+01 1.4000e+01]
 [       nan        nan 1.1872e+04 ... 2.3000e+01 2.9000e+01 1.3000e+01]
 ...
 [       nan        nan        nan ... 1.1189e+04 4.1870e+03 2.2670e+03]
 [       nan        nan        nan ...        nan 1.2808e+04 3.7670e+03]
 [       nan        nan        nan ...        nan        nan 1.4430e+04]]


<div class=warn>
Be careful since matrices for big genomic regions can be quite heavy in memory.
</div>

## Annotating a PixelTable

`PixelTable` provides methods to add annotations to the bins and the pixels, `add_bin_annotation` and `add_pix_annotation` respectively. These methods are particularly useful when you do not want to pollute your cooler file with annotations you will use just a couple of times. `add_bin_annotation` works in the same way as `HiconaCooler.bin_annotation_add`, see the `HiconaCooler` notebook for examples. `add_pix_annotation` instead works by simply performing a join between the pixels and the provided annotation on the columns `bin1_id` and `bin2_id`.

In [10]:
# Bin annotation
hmm_states = pl.read_csv(HMM_STATES, separator="\t", new_columns=("chrom", "start", "end", "state"))

table.add_bin_annotation(
    hmm_states, 
    metric="chrom_enrich", 
    consolidate=True, 
    save_all_mods=True, 
    ignore_null_mode=False,
)

table.bins.get_dataframe("chr17:0-1000000").head()

bin_id,chrom,start,end,weight,null_chrom_enrich,Repr_Pc_chrom_enrich,Enh_chrom_enrich,Het_chrom_enrich,Void_chrom_enrich,Prom_chrom_enrich,Tx_chrom_enrich
i64,str,i32,i32,f64,f64,f64,f64,f64,f64,f64,f64
26475,"""chr17""",0,100000,null,4.034009,null,null,null,null,null,null
26476,"""chr17""",100000,200000,0.007338,3.124769,1.487346,0.035446,0.286166,null,null,null
26477,"""chr17""",200000,300000,0.006855,null,2.029982,0.070042,1.360146,0.425676,null,null
26478,"""chr17""",300000,400000,0.005692,null,1.928669,0.903479,1.428936,0.263276,0.788412,0.058686
26479,"""chr17""",400000,500000,0.007215,3.034053,0.61664,0.508783,0.616187,0.893754,0.719294,0.52198


In [11]:
# Pixel annotation
randints = pl.read_csv(RANDINTS, separator="\t")  # Random integer for each pixel

table.add_pix_annotation(randints)
table.get_dataframe("chr17:0-1000000").head()

bin1_id,bin2_id,count,rand_int
i64,i64,i32,i64
26475,26475,17,73
26475,26481,1,47
26476,26476,7035,11
26476,26477,4003,2
26476,26478,3392,68


## Persistent PixelTable

We introduced the `PixelTable` class saying that it creates temporary copies of the data. Sometimes though, you might want to save your current table, maybe because performing clustering on it took a long time and you do not want to repeat it every time. To do so, `PixelTable` provides two methods, `save` and `load`. 

<div class=note>
    `save` creates a permanent snapshot of the `PixelTable` at some provided path; further changes of the `PixelTable` do not affect the 
    saved snapshot. `load` creates a new `PixelTable` instance whose data is backed by a copy of the snapshot in the temporary folder.
    This means that changes on the loaded object do not affect the snapshot and must be saved to a new snapshot if needed.
</div>

In [12]:
with tempfile.TemporaryDirectory() as tmp_dir_name:
    snapshot_path = os.path.join(tmp_dir_name, "test_table")

    table.save(snapshot_path)
    reloaded = hi.PixelTable.load(snapshot_path)

    # The reloaded table preserves the annotations which were previously added
    print(reloaded.get_dataframe("chr17:0-10000000").head())


shape: (5, 4)
┌─────────┬─────────┬───────┬──────────┐
│ bin1_id ┆ bin2_id ┆ count ┆ rand_int │
│ ---     ┆ ---     ┆ ---   ┆ ---      │
│ i64     ┆ i64     ┆ i32   ┆ i64      │
╞═════════╪═════════╪═══════╪══════════╡
│ 26475   ┆ 26475   ┆ 17    ┆ 73       │
│ 26475   ┆ 26481   ┆ 1     ┆ 47       │
│ 26475   ┆ 26518   ┆ 1     ┆ 53       │
│ 26475   ┆ 26526   ┆ 1     ┆ 92       │
│ 26475   ┆ 26534   ┆ 1     ┆ 95       │
└─────────┴─────────┴───────┴──────────┘


In [13]:
from IPython.core.display import HTML

def css_styling():
    styles = open("../styles/custom.css", "r").read()
    return HTML(styles)
css_styling()
